# Chapter 6 — AI Governance & Zero Trust for Agents

## Objective

Turn governance frameworks into runnable checks — a scored checklist, a RACI, a Zero-Trust audit with a hard go-live gate, and an ordered remediation plan.

By the end you will be able to:

- Score a system against a blended governance checklist
- Explain weighted scoring and severity ranking of gaps
- Run a Zero-Trust audit and apply the audit-trail exit criterion
- Produce an ordered, critical-first remediation plan
- Read an AI RACI matrix and identify the accountable owner

## Summary

Governance is where frameworks become code that can block a release. This notebook builds a small governance assistant: it scores an agentic system against a blended checklist that maps every control to a risk-management function, emits an accountability (RACI) matrix, runs a Zero-Trust audit whose exit criterion is 'every component is logged,' and produces an ordered, critical-first remediation plan. It is pure standard-library Python you can read end to end. The central idea is an asymmetry: a high governance score with a missing audit trail still blocks go-live, because if an action was not logged, you cannot prove what happened.

## Outline

**Chapter arc:** regulatory & compliance landscape → NIST AI RMF & the GenAI Profile → OWASP LLM governance checklist → Zero Trust for agents → post-quantum & advanced threats

This notebook covers:

- The blended governance checklist
- Scoring the gaps
- The AI RACI matrix
- The Zero-Trust audit and go-live gate
- The remediation plan
- Runnable demos
- A self-check quiz

## The big picture

When an agent acts on its own, **liability shifts to the operator** — so governance becomes code that can block a release.

- **Regulation:** the EU AI Act sorts systems into risk tiers (unacceptable → high → limited → minimal); most customer-facing agents carry at least transparency duties, and high-risk uses add logging and human oversight.
- **Frameworks to controls:** NIST AI RMF (Govern / Map / Measure / Manage) and the OWASP LLM governance checklist (six building blocks).
- **Zero Trust for agents** (NIST SP 800-207): the model, its tools, and its data are untrusted by default — verify every request, enforce least privilege, assume breach.
- **The determinism sandwich:** wrap the probabilistic model in a deterministic input boundary and a deterministic output boundary. The audit trail is the hard go-live gate.

**The determinism sandwich**

<svg viewBox="0 0 760 250" width="100%" style="max-width:760px;height:auto;font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif" xmlns="http://www.w3.org/2000/svg" role="img"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs><rect width="760" height="250" rx="12" fill="#ffffff"/><rect x="22" y="22" width="716" height="56" rx="9" fill="#0f766e"/><text x="38" y="50" fill="#fff" font-size="15" font-weight="600" dominant-baseline="middle">Deterministic input</text><text x="722" y="50" fill="#e8edf5" font-size="12" text-anchor="end" dominant-baseline="middle">validate · sanitize · authorize</text><rect x="22" y="88" width="716" height="56" rx="9" fill="#a16207"/><text x="38" y="116" fill="#fff" font-size="15" font-weight="600" dominant-baseline="middle">Probabilistic model</text><text x="722" y="116" fill="#e8edf5" font-size="12" text-anchor="end" dominant-baseline="middle">the LLM — capable, never trusted to police itself</text><rect x="22" y="154" width="716" height="56" rx="9" fill="#1d4ed8"/><text x="38" y="182" fill="#fff" font-size="15" font-weight="600" dominant-baseline="middle">Deterministic output</text><text x="722" y="182" fill="#e8edf5" font-size="12" text-anchor="end" dominant-baseline="middle">schema-check · policy · human-gate · sign to audit trail</text><text x="380" y="232" fill="#334155" font-size="12.5" text-anchor="middle">Wrap the untrusted model in verified input and output boundaries — Zero Trust for the agent.</text></svg>

## Setup

**Pure Python standard library** — no installs, no API key, no network. Everything runs as-is.

Imports used across the examples:

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import IntEnum

## Core building blocks

Each function below is introduced, then defined. Run them all, then see them in action in **Try it** and the **Live demo**.

### The blended governance checklist

A blended governance checklist: each Control maps to a risk-management function and a governance block, carries a weight, and may be marked critical (a hard gate on go-live).

> **Security lens:** each control maps to a NIST AI RMF function and an OWASP block; critical controls gate go-live regardless of the numeric score.

> **Real-world case — Italy's Garante v. ChatGPT (2023).** Italy's regulator ordered an emergency stop over no lawful basis, no age check, and transparency failures, later fining OpenAI €15M (annulled on appeal in 2026). Deploy with lawful basis, transparency, and oversight from day one.

In [2]:
class Severity(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4


@dataclass(frozen=True)
class Control:
    cid: str
    text: str
    rmf: str          # NIST AI RMF function: Govern | Map | Measure | Manage
    owasp: str        # OWASP checklist block
    weight: int       # 1..3
    critical: bool = False


CHECKLIST = [
    Control("GOV-1", "Named accountable owner + AI RACI for the agent", "Govern", "Governance", 3, critical=True),
    Control("GOV-2", "AI use policy, acceptable-use + legal/regulatory sign-off", "Govern", "Legal", 2),
    Control("MAP-1", "AI asset inventory (SBOM + MLBOM) is current", "Map", "Asset Inventory", 3, critical=True),
    Control("MAP-2", "Threat model exists (MITRE ATLAS) incl. agent abuse cases", "Map", "Threat Modeling", 3),
    Control("MAP-3", "Data provenance + lawful basis (GDPR Art.6/22) documented", "Map", "Legal", 2),
    Control("MEA-1", "TEVV: pre-deployment eval + red-team results recorded", "Measure", "Testing & Eval", 3),
    Control("MEA-2", "KPIs tracked (refusal rate, injection-block rate, MTTD)", "Measure", "Testing & Eval", 2),
    Control("MAN-1", "Every agent action emits a signed audit event", "Manage", "Deployment", 3, critical=True),
    Control("MAN-2", "Human-in/on-the-loop gate on irreversible actions", "Manage", "Deployment", 3, critical=True),
    Control("MAN-3", "Least-privilege agent identity; no shared god-credential", "Manage", "Deployment", 3),
    Control("MAN-4", "Incident response + rollback runbook for the agent", "Manage", "Deployment", 2),
    Control("ZT-1", "Input validation (prompt/RAG/tool) as micro-segmentation", "Manage", "Deployment", 2),
    Control("PQC-1", "Crypto-agility plan toward PQC (FIPS 203/204/205)", "Govern", "Strategy", 1),
]

### Scoring the gaps

Score a described system against the checklist — met, partial, or missing — returning a 0–100 score plus the open gaps ranked by severity.

> **Security lens:** a weighted score plus severity-ranked gaps turns 'are we compliant?' into a function you can run in CI.

In [3]:
@dataclass
class Gap:
    control: Control
    state: str
    severity: Severity


def _severity(ctrl: Control, state: str) -> Severity:
    if state == "met":
        return Severity.LOW
    base = {1: Severity.LOW, 2: Severity.MEDIUM, 3: Severity.HIGH}[ctrl.weight]
    if ctrl.critical and state == "missing":
        return Severity.CRITICAL
    if state == "partial":            # half a step down from missing
        base = Severity(max(Severity.LOW, base - 1))
    return base


def gap_analysis(system_profile: dict) -> dict:
    """Return {'score': 0..100, 'gaps': [Gap...], 'critical_open': int}."""
    gaps, earned, possible = [], 0, 0
    for c in CHECKLIST:
        possible += c.weight
        state = system_profile.get(c.cid, "missing")  # unscored controls default to "missing"
        # Scoring: a met control earns full weight, partial earns half, missing earns nothing.
        earned += {"met": c.weight, "partial": c.weight / 2, "missing": 0}[state]
        if state != "met":
            gaps.append(Gap(c, state, _severity(c, state)))
    gaps.sort(key=lambda g: (-g.severity, g.control.cid))
    return {
        "score": round(100 * earned / possible),
        "gaps": gaps,
        "critical_open": sum(1 for g in gaps if g.severity == Severity.CRITICAL),
    }

### The AI RACI matrix

The accountability matrix: who is Responsible, Accountable, Consulted, and Informed across the governance activities.

> **Security lens:** a RACI exists to kill the 'who owns AI risk?' ambiguity — exactly one accountable owner per activity.

In [4]:
RACI_ROLES = ["Product Owner", "ML Eng", "Security", "Legal/Privacy", "SRE/Ops"]
RACI_ACTIVITIES = {
    "Define AI use policy":        ["C", "I", "C", "A", "I"],
    "Maintain asset inventory":    ["A", "R", "C", "I", "R"],
    "Threat model + red team":     ["I", "C", "A/R", "C", "I"],
    "Approve go-live":             ["A", "C", "R", "R", "C"],
    "Monitor + audit in prod":     ["I", "C", "A", "I", "R"],
    "Incident response":           ["C", "R", "A", "C", "R"],
}


def build_raci() -> dict:
    """Return {'roles': [...], 'rows': {activity: [codes...]}}."""
    return {"roles": RACI_ROLES, "rows": RACI_ACTIVITIES}

### The Zero-Trust audit and go-live gate

`zero_trust_audit(components)` is the **differentiator**. For each component it checks four Zero Trust properties — distinct **identity**, **least privilege**, a signed **audit trail**, and a **human gate** on irreversible actions — and records findings.

> **Security lens:** if it isn't logged, it didn't happen — and you can't govern it. A missing audit trail blocks go-live even when the score is high.

> **Real-world case — Air Canada (Moffatt, 2024).** A tribunal held the airline liable for wrong advice its website chatbot gave, rejecting the claim that the bot was a separate legal entity. The damages were tiny (CA$650.88) but the precedent is large: no oversight, no audit trail, no way to disown an agent's output.

In [5]:
def zero_trust_audit(components: list) -> dict:
    findings, blocked = [], False
    for comp in components:
        issues = []
        if not comp.get("audit_trail"):
            issues.append(("CRITICAL", "no signed audit trail (ZT exit criterion)"))
            blocked = True
        if not comp.get("has_identity"):
            issues.append(("HIGH", "no distinct authenticated identity"))
        if not comp.get("least_privilege"):
            issues.append(("HIGH", "over-privileged (no least privilege)"))
        if comp.get("irreversible") and not comp.get("human_gate"):
            issues.append(("HIGH", "irreversible action with no human gate"))
        findings.append({"component": comp["name"], "issues": issues})
    return {"go_live_blocked": blocked, "findings": findings}

### The remediation plan

Merge the open checklist gaps and the Zero-Trust findings into one ordered to-do list, critical items first.

> **Security lens:** each remediation line carries its RMF function and control id, so a finding maps straight back to EU AI Act, NIST RMF, and GDPR duties.

In [6]:
def remediation_plan(gap_result: dict, zt_result: dict) -> list:
    steps = []
    # Zero Trust CRITICALs lead the plan. An unlogged component here can also surface as
    # the MAN-1 checklist control below -- the same root cause seen through two lenses.
    for f in zt_result["findings"]:
        for sev, msg in f["issues"]:
            if sev == "CRITICAL":
                steps.append((Severity.CRITICAL, f"[{f['component']}] {msg}"))
    for g in gap_result["gaps"]:
        steps.append((g.severity, f"[{g.control.cid}/{g.control.rmf}] {g.control.text}"))
    steps.sort(key=lambda s: -s[0])
    return [f"{i+1}. ({sev.name}) {msg}" for i, (sev, msg) in enumerate(steps)]

## Try it

Run the self-contained demo — the examples that need no external service — and read the output.

In [7]:
# A partially-governed agent: good detection story, weak audit/identity.
profile = {
    "GOV-1": "met", "GOV-2": "partial", "MAP-1": "partial", "MAP-2": "met",
    "MAP-3": "missing", "MEA-1": "met", "MEA-2": "partial", "MAN-1": "missing",
    "MAN-2": "met", "MAN-3": "partial", "MAN-4": "met", "ZT-1": "met", "PQC-1": "missing",
}
components = [
    {"name": "planner-agent",  "has_identity": True,  "least_privilege": True,
     "audit_trail": True,  "irreversible": False, "human_gate": True},
    {"name": "executor-agent", "has_identity": True,  "least_privilege": False,
     "audit_trail": False, "irreversible": True,  "human_gate": True},   # CRITICAL: no audit
    {"name": "tool-gateway",   "has_identity": False, "least_privilege": True,
     "audit_trail": True,  "irreversible": False, "human_gate": False},
]

ga = gap_analysis(profile)
zt = zero_trust_audit(components)

print("=== AI Governance Assistant ===")
print(f"Governance score: {ga['score']}/100   "
      f"(open critical controls: {ga['critical_open']})")
print(f"Zero Trust go-live: {'BLOCKED' if zt['go_live_blocked'] else 'CLEARED'}\n")

print("Top gaps:")
for g in ga["gaps"][:5]:
    print(f"  - ({g.severity.name:8}) {g.control.cid}: {g.control.text}")

print("\nAI RACI (Approve go-live):", dict(zip(RACI_ROLES, build_raci()["rows"]["Approve go-live"])))

print("\nRemediation plan:")
for line in remediation_plan(ga, zt)[:6]:
    print("  " + line)

# Derive the blocker from the live result so this trailer never lies if the
# sample data changes (the audit trail is the Zero Trust go-live exit criterion).
unlogged = [f["component"] for f in zt["findings"]
            if any("audit trail" in msg for _, msg in f["issues"])]
if unlogged:
    print(f"\nExit criterion: go-live is BLOCKED until every component is logged "
          f"(NIST SP 800-207). '{unlogged[0]}' has no audit trail.")
else:
    print("\nExit criterion: MET -- every component is logged; go-live is CLEARED "
          "(NIST SP 800-207).")

=== AI Governance Assistant ===
Governance score: 66/100   (open critical controls: 1)
Zero Trust go-live: BLOCKED

Top gaps:
  - (CRITICAL) MAN-1: Every agent action emits a signed audit event
  - (MEDIUM  ) MAN-3: Least-privilege agent identity; no shared god-credential
  - (MEDIUM  ) MAP-1: AI asset inventory (SBOM + MLBOM) is current
  - (MEDIUM  ) MAP-3: Data provenance + lawful basis (GDPR Art.6/22) documented
  - (LOW     ) GOV-2: AI use policy, acceptable-use + legal/regulatory sign-off

AI RACI (Approve go-live): {'Product Owner': 'A', 'ML Eng': 'C', 'Security': 'R', 'Legal/Privacy': 'R', 'SRE/Ops': 'C'}

Remediation plan:
  1. (CRITICAL) [executor-agent] no signed audit trail (ZT exit criterion)
  2. (CRITICAL) [MAN-1/Manage] Every agent action emits a signed audit event
  3. (MEDIUM) [MAN-3/Manage] Least-privilege agent identity; no shared god-credential
  4. (MEDIUM) [MAP-1/Map] AI asset inventory (SBOM + MLBOM) is current
  5. (MEDIUM) [MAP-3/Map] Data provenance + lawful 

## Quiz

Answer each question, then fill in the `answers` dict in the next cell (`"A"`, `"B"`, …) and run it to grade yourself.

**1. In zero_trust_audit, what single condition blocks go-live regardless of the score?**

- A. A low weight
- B. Any component with no signed audit trail
- C. A missing RACI
- D. A slow response

**2. In gap_analysis scoring, a 'partial' control earns how much of its weight?**

- A. Full
- B. Half
- C. Zero
- D. Double

**3. A control that is both critical and missing is ranked at which severity?**

- A. LOW
- B. MEDIUM
- C. HIGH
- D. CRITICAL

**4. The notebook models two independent go-live gates. What are they?**

- A. Speed and cost
- B. The component-level Zero Trust audit and the process-level governance checklist
- C. Identity and encryption
- D. Logging and metrics

**5. In an AI RACI matrix, how many roles are Accountable (A) for a given activity?**

- A. Zero
- B. Exactly one
- C. At least two
- D. All of them


In [8]:
import base64
_KEY = {'q1': 'Qg==', 'q2': 'Qg==', 'q3': 'RA==', 'q4': 'Qg==', 'q5': 'Qg=='}   # base64 so the answers aren't obvious at a glance
_WHY = {'q1': 'The audit-trail exit criterion flips go_live_blocked and keeps it flipped.', 'q2': 'met = full, partial = half, missing = zero.', 'q3': 'Missing + critical is escalated to CRITICAL.', 'q4': 'Wiring logging (engineering) is not the same as an owner signing off (governance).', 'q5': "Exactly one accountable owner removes the 'who owns it?' ambiguity."}

def check(answers):
    key = {q: base64.b64decode(v).decode() for q, v in _KEY.items()}
    if all(str(answers.get(q, '')).strip().upper() in ('', '?') for q in key):
        print('Fill in the answers dict above (A/B/C/D) and re-run to grade.')
        return
    score = 0
    for q in sorted(key):
        got = str(answers.get(q, '')).strip().upper()
        if got == key[q]:
            score += 1; print(f'\u2713 {q}: {got}')
        else:
            print(f'\u2717 {q}: you put {got or "\u2014"} \u2014 correct is {key[q]}. {_WHY[q]}')
    print(f'\nScore: {score}/{len(key)}')

In [9]:
answers = {
    "q1": "?",   # A / B / C / D
    "q2": "?",   # A / B / C / D
    "q3": "?",   # A / B / C / D
    "q4": "?",   # A / B / C / D
    "q5": "?",   # A / B / C / D
}
check(answers)

Fill in the answers dict above (A/B/C/D) and re-run to grade.


## Key takeaways

- Governance is a deployable control layer — data plus a few functions that can block a release, not paperwork.
- Exactly one control is a hard gate: no signed audit trail blocks go-live regardless of the score.
- Component-level Zero Trust and process-level checklists are two independent gates; mature programs pass both.
- A RACI kills the 'who owns AI risk?' ambiguity — exactly one accountable owner per activity.